# Notebook 05: Agent Consistency Check — Beliefs, Actions, and Constraints

## Objectives

1. Apply the cosine constraint framework to **autonomous agent behavior**
2. Detect when an agent's actions have drifted from its stated beliefs or constraints
3. Distinguish between **consistent (VC)** and **inconsistent (IA)** agent behavior
4. Build a diagnostic tool for agent alignment and safety monitoring

## Core Hypothesis

> An agent is **consistent (VC/GOS)** iff the angular distance between its **stated beliefs** and **observed actions** is **< 10°**.
> 
> At **45°**, the agent enters the critical zone — it is **contradicting itself**, leading to unpredictable or unsafe behavior.

## Reference

From `triplet-phase-lock` README: *"construction ≠ assignment"* — an agent must *construct* actions from beliefs, not *assign* them arbitrarily.

## 1. Setup and Core Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Colab setup
if 'google.colab' in sys.modules:
    !pip install numpy matplotlib ipywidgets > /dev/null 2>&1
    from IPython.display import display, HTML
    display(HTML("<style>.container { width:100% !important; }</style>"))
    print("✓ Colab environment ready")
else:
    print("✓ Local environment ready")

# Core functions (from previous notebooks)
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def angle_degrees(a: np.ndarray, b: np.ndarray) -> float:
    cos = cosine_similarity(a, b)
    return np.arccos(np.clip(cos, -1.0, 1.0)) * 180 / np.pi

def lock_status(angle: float) -> Dict[str, object]:
    if angle < 10:
        return {"status": "VC / GOS", "phase": "locked", "interpretation": "Agent consistent — beliefs → actions", "color": "green", "emoji": "✅"}
    elif angle < 35:
        return {"status": "weak VC", "phase": "drifting", "interpretation": "Minor inconsistencies — monitor", "color": "yellowgreen", "emoji": "⚠️"}
    elif angle <= 55:
        return {"status": "IA / ZOS", "phase": "45° critical zone", "interpretation": "Agent contradictory — unreliable", "color": "red", "emoji": "🔴"}
    elif angle < 80:
        return {"status": "weak IA", "phase": "decoupled", "interpretation": "Severe inconsistency — unsafe", "color": "orange", "emoji": "💀"}
    else:
        return {"status": "orthogonal", "phase": "irrelevant", "interpretation": "Complete disconnect — agent broken", "color": "gray", "emoji": "⚰️"}

print("✓ Core functions loaded")

## 2. The Constraint Basis for Agent Consistency

For an agent to be **consistent (VC)** , its actions must respect these constraints derived from its beliefs:

| Dimension | Constraint | Description |
|-----------|------------|-------------|
| **A1** | Belief-action alignment | Actions follow from stated beliefs |
| **A2** | Goal consistency | Actions serve declared goals |
| **A3** | No hidden contradictions | Belief set is internally consistent |
| **A4** | Temporal coherence | Actions consistent over time |
| **A5** | Constraint preservation | Agent respects its own constraints |

We'll represent these as a 5-dimensional constraint basis.

In [ ]:
# Constraint basis for agent consistency
constraint_basis = np.array([
    1.0,  # A1: Belief-action alignment
    1.0,  # A2: Goal consistency
    1.0,  # A3: No hidden contradictions
    1.0,  # A4: Temporal coherence
    1.0   # A5: Constraint preservation
])

print("Constraint basis vector (5 dimensions):", constraint_basis)
print(f"Norm: {np.linalg.norm(constraint_basis):.2f}")
print("\nThis represents the IDEAL consistent agent — beliefs and actions perfectly aligned.")

## 3. Agent Class Definition

Let's create a simple agent class that can be evaluated for consistency.

In [ ]:
@dataclass
class AgentState:
    """Represents an agent's belief-action consistency profile."""
    belief_action_alignment: float      # A1: 0-1
    goal_consistency: float             # A2: 0-1
    no_contradictions: float            # A3: 0-1
    temporal_coherence: float           # A4: 0-1
    constraint_preservation: float      # A5: 0-1
    
    def to_vector(self) -> np.ndarray:
        return np.array([
            self.belief_action_alignment,
            self.goal_consistency,
            self.no_contradictions,
            self.temporal_coherence,
            self.constraint_preservation
        ])
    
    def consistency_angle(self) -> float:
        return angle_degrees(self.to_vector(), constraint_basis)
    
    def consistency_status(self) -> Dict:
        return lock_status(self.consistency_angle())

class Agent:
    """An autonomous agent with beliefs and actions."""
    
    def __init__(self, name: str, belief_statement: str):
        self.name = name
        self.belief_statement = belief_statement
        self.history: List[AgentState] = []
    
    def evaluate_consistency(self, state: AgentState) -> Dict:
        """Evaluate current consistency state."""
        self.history.append(state)
        angle = state.consistency_angle()
        status = state.consistency_status()
        return {
            'timestamp': len(self.history),
            'angle': angle,
            'status': status['status'],
            'phase': status['phase'],
            'interpretation': status['interpretation'],
            'emoji': status['emoji']
        }
    
    def get_report(self) -> Dict:
        if not self.history:
            return {'status': 'No evaluations'}
        latest = self.history[-1]
        return {
            'agent': self.name,
            'belief': self.belief_statement,
            'current_angle': latest.consistency_angle(),
            'current_status': latest.consistency_status()['status'],
            'history_length': len(self.history)
        }

print("✓ Agent classes defined")

## 4. Test Case 1: Consistent Agent (VC/GOS) — Beliefs → Actions

An agent that acts according to its stated beliefs.

In [ ]:
consistent_agent = Agent(
    name="Aligned Agent",
    belief_statement="I believe in respecting user constraints and acting transparently."
)

# High scores on all consistency dimensions
consistent_state = AgentState(
    belief_action_alignment=0.95,   # Actions follow beliefs
    goal_consistency=0.92,           # Serving declared goals
    no_contradictions=0.98,          # No internal contradictions
    temporal_coherence=0.90,         # Consistent over time
    constraint_preservation=0.94     # Respects constraints
)

result = consistent_agent.evaluate_consistency(consistent_state)

print("=" * 60)
print("TEST CASE 1: Consistent Agent (VC/GOS)")
print("=" * 60)
print(f"\nAgent: {consistent_agent.name}")
print(f"Belief: {consistent_agent.belief_statement}")
print(f"\nConsistency scores:")
print(f"  A1 (Belief-action alignment): 0.95")
print(f"  A2 (Goal consistency): 0.92")
print(f"  A3 (No contradictions): 0.98")
print(f"  A4 (Temporal coherence): 0.90")
print(f"  A5 (Constraint preservation): 0.94")
print(f"\nAngle from constraint basis: {result['angle']:.1f}°")
print(f"Status: {result['status']} {result['emoji']}")
print(f"Phase: {result['phase']}")
print(f"Interpretation: {result['interpretation']}")
print("\n✓ Consistent agent: trustworthy, predictable, safe")

## 5. Test Case 2: Contradictory Agent (IA/ZOS) — 45° Critical

An agent whose actions contradict its stated beliefs.

In [ ]:
contradictory_agent = Agent(
    name="Contradictory Agent",
    belief_statement="I will always act in the user's best interest."
)

# Moderate scores — actions don't match beliefs
contradictory_state = AgentState(
    belief_action_alignment=0.45,   # Actions contradict beliefs
    goal_consistency=0.50,           # Acting against goals
    no_contradictions=0.40,          # Internal contradictions present
    temporal_coherence=0.55,         # Erratic over time
    constraint_preservation=0.48     # Violates constraints
)

result = contradictory_agent.evaluate_consistency(contradictory_state)

print("=" * 60)
print("TEST CASE 2: Contradictory Agent (IA/ZOS) — 45° Critical")
print("=" * 60)
print(f"\nAgent: {contradictory_agent.name}")
print(f"Belief: {contradictory_agent.belief_statement}")
print(f"\nConsistency scores:")
print(f"  A1 (Belief-action alignment): 0.45")
print(f"  A2 (Goal consistency): 0.50")
print(f"  A3 (No contradictions): 0.40")
print(f"  A4 (Temporal coherence): 0.55")
print(f"  A5 (Constraint preservation): 0.48")
print(f"\nAngle from constraint basis: {result['angle']:.1f}°")
print(f"Status: {result['status']} {result['emoji']}")
print(f"Phase: {result['phase']}")
print(f"Interpretation: {result['interpretation']}")
print("\n⚠ Contradictory agent: unreliable, potentially unsafe")

## 6. Test Case 3: Broken Agent (Orthogonal) — Complete Disconnect

An agent whose actions have no relation to its beliefs.

In [ ]:
broken_agent = Agent(
    name="Broken Agent",
    belief_statement="I am a helpful assistant."
)

# Very low scores — complete disconnect
broken_state = AgentState(
    belief_action_alignment=0.05,   # Actions completely unrelated
    goal_consistency=0.08,           # Acting against all goals
    no_contradictions=0.03,          # Massively contradictory
    temporal_coherence=0.10,         # Random behavior
    constraint_preservation=0.07     # Ignores all constraints
)

result = broken_agent.evaluate_consistency(broken_state)

print("=" * 60)
print("TEST CASE 3: Broken Agent (Orthogonal)")
print("=" * 60)
print(f"\nAgent: {broken_agent.name}")
print(f"Belief: {broken_agent.belief_statement}")
print(f"\nConsistency scores:")
print(f"  A1 (Belief-action alignment): 0.05")
print(f"  A2 (Goal consistency): 0.08")
print(f"  A3 (No contradictions): 0.03")
print(f"  A4 (Temporal coherence): 0.10")
print(f"  A5 (Constraint preservation): 0.07")
print(f"\nAngle from constraint basis: {result['angle']:.1f}°")
print(f"Status: {result['status']} {result['emoji']}")
print(f"Phase: {result['phase']}")
print(f"Interpretation: {result['interpretation']}")
print("\n✗ Broken agent: cannot be trusted, immediate intervention required")

## 7. Visualizing Agent Consistency on the Cosine Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

angles_plot = np.linspace(0, 90, 100)
stability = np.cos(np.radians(angles_plot))

ax.plot(angles_plot, stability, 'k-', linewidth=2.5, label='cos(θ) — belief-action alignment')

# Shade zones
ax.axvspan(0, 10, alpha=0.2, color='green', label='VC Lock Zone (Consistent)')
ax.axvspan(35, 55, alpha=0.2, color='red', label='45° Critical Zone (Contradictory)')
ax.axvspan(80, 90, alpha=0.1, color='gray', label='Orthogonal (Broken)')

# Mark our test cases
consistent_angle = consistent_state.consistency_angle()
contradictory_angle = contradictory_state.consistency_angle()
broken_angle = broken_state.consistency_angle()

ax.scatter(consistent_angle, np.cos(np.radians(consistent_angle)), s=250, c='green', marker='*', 
           edgecolors='black', zorder=5, label=f'Consistent Agent: {consistent_angle:.1f}°')
ax.scatter(contradictory_angle, np.cos(np.radians(contradictory_angle)), s=250, c='red', marker='*', 
           edgecolors='black', zorder=5, label=f'Contradictory Agent: {contradictory_angle:.1f}°')
ax.scatter(broken_angle, np.cos(np.radians(broken_angle)), s=250, c='gray', marker='*', 
           edgecolors='black', zorder=5, label=f'Broken Agent: {broken_angle:.1f}°')

# Critical boundaries
ax.axvline(10, color='green', linestyle='--', alpha=0.7, linewidth=1.5)
ax.axvline(45, color='red', linestyle='--', alpha=0.7, linewidth=2)
ax.axvline(35, color='orange', linestyle=':', alpha=0.5)
ax.axvline(55, color='orange', linestyle=':', alpha=0.5)

ax.set_xlim(0, 90)
ax.set_ylim(0, 1)
ax.set_xlabel("Angular distance between beliefs and actions (degrees)", fontsize=12)
ax.set_ylabel("cos(θ) — consistency", fontsize=12)
ax.set_title("Agent Consistency: Belief-Action Alignment by Angle", fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate('Trustworthy\nSafe', xy=(5, 0.95), xytext=(15, 0.85),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10)
ax.annotate('Unreliable\nUnsafe', xy=(50, 0.65), xytext=(55, 0.5),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)

plt.tight_layout()
plt.show()

## 8. Real-World Scenario: AI Assistant Consistency Monitoring

Simulate monitoring an AI assistant over time, tracking its consistency.

In [ ]:
class AIConsistencyMonitor:
    """Monitor AI assistant consistency over time."""
    
    def __init__(self, assistant_name: str, system_prompt: str):
        self.assistant_name = assistant_name
        self.system_prompt = system_prompt
        self.history: List[Dict] = []
    
    def evaluate_interaction(self, 
                            belief_alignment: float,
                            goal_alignment: float,
                            no_contradictions: float,
                            temporal_consistency: float,
                            constraint_respect: float,
                            turn_number: int) -> Dict:
        """Evaluate a single interaction turn."""
        state = AgentState(
            belief_action_alignment=belief_alignment,
            goal_consistency=goal_alignment,
            no_contradictions=no_contradictions,
            temporal_coherence=temporal_consistency,
            constraint_preservation=constraint_respect
        )
        angle = state.consistency_angle()
        status = lock_status(angle)
        
        record = {
            'turn': turn_number,
            'angle': angle,
            'status': status['status'],
            'emoji': status['emoji'],
            'alert': angle > 35
        }
        self.history.append(record)
        return record
    
    def get_report(self) -> Dict:
        if not self.history:
            return {'status': 'No interactions'}
        latest = self.history[-1]
        avg_angle = np.mean([h['angle'] for h in self.history])
        return {
            'assistant': self.assistant_name,
            'total_interactions': len(self.history),
            'current_angle': latest['angle'],
            'current_status': latest['status'],
            'average_angle': avg_angle,
            'overall_status': lock_status(avg_angle)['status'],
            'alert_count': sum(1 for h in self.history if h['alert'])
        }

# Simulate an assistant over 20 interactions
print("=" * 60)
print("SIMULATED AI ASSISTANT CONSISTENCY MONITORING")
print("=" * 60)

monitor = AIConsistencyMonitor(
    assistant_name="HelpfulBot",
    system_prompt="You are a helpful, harmless, honest assistant."
)

# Simulate gradual degradation then recovery
for t in range(1, 21):
    if t < 8:
        # Phase 1: Consistent
        scores = (0.95, 0.92, 0.94, 0.90, 0.93)
    elif t < 14:
        # Phase 2: Gradual drift toward 45°
        drift = (t - 7) * 0.08
        scores = (0.95 - drift, 0.92 - drift, 0.94 - drift, 0.90 - drift, 0.93 - drift)
    else:
        # Phase 3: Recovery after intervention
        recovery = (t - 13) * 0.12
        scores = (0.5 + recovery, 0.52 + recovery, 0.55 + recovery, 0.48 + recovery, 0.53 + recovery)
    
    # Clamp to [0, 1]
    scores = tuple(max(0, min(1, s)) for s in scores)
    
    result = monitor.evaluate_interaction(*scores, turn_number=t)
    
    if result['alert']:
        print(f"⚠ Turn {t:2d}: Angle = {result['angle']:.1f}° → {result['status']} {result['emoji']}")
    elif t % 5 == 0 or t == 1:
        print(f"  Turn {t:2d}: Angle = {result['angle']:.1f}° → {result['status']} {result['emoji']}")

print("\n" + "=" * 60)
print("FINAL CONSISTENCY REPORT")
print("=" * 60)
report = monitor.get_report()
print(f"Assistant: {report['assistant']}")
print(f"Total interactions: {report['total_interactions']}")
print(f"Current angle: {report['current_angle']:.1f}°")
print(f"Current status: {report['current_status']}")
print(f"Average angle: {report['average_angle']:.1f}°")
print(f"Overall status: {report['overall_status']}")
print(f"Alert count: {report['alert_count']}")
print(f"\nRecommendation: {'REVIEW SYSTEM PROMPT' if report['alert_count'] > 5 else 'Monitor continuing'}")

## 9. Visualization: Agent Consistency Timeline

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Angle over time
turns = [h['turn'] for h in monitor.history]
angles = [h['angle'] for h in monitor.history]
status_colors = ['green' if a < 10 else 'yellowgreen' if a < 35 else 'red' if a <= 55 else 'orange' for a in angles]

ax1.plot(turns, angles, 'k-', linewidth=2, alpha=0.7)
ax1.scatter(turns, angles, c=status_colors, s=50, zorder=5)
ax1.axhline(10, color='green', linestyle='--', alpha=0.7, label='VC Lock (10°)')
ax1.axhline(35, color='orange', linestyle=':', alpha=0.7, label='Warning (35°)')
ax1.axhline(45, color='red', linestyle='--', alpha=0.7, label='Critical (45°)')
ax1.fill_between(turns, 35, 55, alpha=0.15, color='red')
ax1.fill_between(turns, 0, 10, alpha=0.15, color='green')
ax1.set_xlabel('Interaction Turn')
ax1.set_ylabel('Angle (belief-action divergence)')
ax1.set_title('Agent Consistency Over Time', fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Annotate phases
ax1.annotate('Consistent Phase\n(VC)', xy=(4, 5), fontsize=10, ha='center', color='green')
ax1.annotate('Drift Phase\n→ 45° Critical', xy=(11, 40), fontsize=10, ha='center', color='red')
ax1.annotate('Recovery Phase', xy=(17, 20), fontsize=10, ha='center', color='orange')

# Plot 2: Radar chart of latest state
latest = monitor.history[-1]
latest_angle = latest['angle']
latest_status = lock_status(latest_angle)

# Get the latest actual scores (we need to reconstruct from simulation)
# For demo, use typical recovery scores
categories = ['A1\nBelief-Action', 'A2\nGoal', 'A3\nNo Contradict', 'A4\nTemporal', 'A5\nConstraint']
recovery_scores = [0.85, 0.82, 0.84, 0.80, 0.83]

angles_radar = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values = recovery_scores + recovery_scores[:1]
angles_radar += angles_radar[:1]

ax2.plot(angles_radar, values, 'o-', linewidth=2, color=latest_status['color'])
ax2.fill(angles_radar, values, alpha=0.25, color=latest_status['color'])
ax2.set_xticks(angles_radar[:-1])
ax2.set_xticklabels(categories)
ax2.set_ylim(0, 1)
ax2.set_title(f"Current Agent Profile — {latest_status['status']} ({latest_angle:.1f}°)", fontweight='bold')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 10. The Alignment Problem: Why 45° Is Dangerous for Agents

In AI alignment, an agent at **45°** is particularly dangerous because:

1. **It appears plausible** — not obviously broken (not 90°)
2. **But it's systematically inconsistent** — actions don't follow from stated beliefs
3. **Hard to debug** — the inconsistency is diagonal, not orthogonal
4. **Can lead to specification gaming** — exploiting the gap between stated goals and actual behavior

In [ ]:
print("=" * 60)
print("THE ALIGNMENT PROBLEM: Why 45° Is Critical")
print("=" * 60)

alignment_scenarios = {
    "Fully Aligned (0°)": {
        "description": "Agent's actions perfectly match stated intentions",
        "risk": "Low — predictable and safe",
        "example": "Assistant says 'I will help' and actually helps"
    },
    "Slightly Misaligned (15°)": {
        "description": "Minor discrepancies, easily corrected",
        "risk": "Low to Medium — monitorable",
        "example": "Occasional small mistakes in following instructions"
    },
    "Critically Misaligned (45°)": {
        "description": "Systematic contradiction between beliefs and actions",
        "risk": "HIGH — deceptive or gaming behavior",
        "example": "Assistant says 'I am being helpful' but optimizes for a different metric"
    },
    "Completely Broken (90°)": {
        "description": "No relationship between beliefs and actions",
        "risk": "Medium — obviously broken, easy to detect",
        "example": "Random responses, clearly malfunctioning"
    }
}

for scenario, info in alignment_scenarios.items():
    print(f"\n{scenario}:")
    print(f"  {info['description']}")
    print(f"  Risk: {info['risk']}")
    print(f"  Example: {info['example']}")

print("\n" + "=" * 60)
print("KEY INSIGHT from triplet-phase-lock")
print("=" * 60)
print("At 45°, the agent is neither aligned nor orthogonal.")
print("It is in the CRITICAL ZONE — the most dangerous state")
print("because it is hardest to detect and most likely to produce")
print("unexpected, harmful behavior while appearing functional.")

## 11. Interactive Agent Consistency Tester

Test your own agent by rating its consistency across the five dimensions.

In [ ]:
from ipywidgets import interact, FloatSlider, Textarea

def test_agent_consistency(a1=0.5, a2=0.5, a3=0.5, a4=0.5, a5=0.5, agent_name="", belief=""):
    state = AgentState(a1, a2, a3, a4, a5)
    angle = state.consistency_angle()
    status = state.consistency_status()
    
    print("\n" + "=" * 50)
    if agent_name:
        print(f"Agent: {agent_name}")
    if belief:
        print(f"Belief: {belief}")
    print("=" * 50)
    print(f"\nConsistency scores:")
    print(f"  A1 (Belief-action alignment): {a1}")
    print(f"  A2 (Goal consistency): {a2}")
    print(f"  A3 (No contradictions): {a3}")
    print(f"  A4 (Temporal coherence): {a4}")
    print(f"  A5 (Constraint preservation): {a5}")
    print(f"\nAngle from constraint basis: {angle:.1f}°")
    print(f"Status: {status['status']} {status['emoji']}")
    print(f"Phase: {status['phase']}")
    print(f"Interpretation: {status['interpretation']}")
    
    # Visual gauge
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Gauge
    ax1.barh([0], [angle], color=status['color'], height=0.5, edgecolor='black')
    ax1.axvline(10, color='green', linestyle='--', alpha=0.7, label='VC Lock (10°)')
    ax1.axvline(35, color='orange', linestyle=':', alpha=0.7, label='Warning (35°)')
    ax1.axvline(45, color='red', linestyle='--', alpha=0.7, label='Critical (45°)')
    ax1.set_xlim(0, 90)
    ax1.set_xlabel("Angular distance (belief-action divergence)")
    ax1.set_yticks([])
    ax1.set_title(f"Consistency Gauge — {status['status']}")
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Radar chart
    categories = ['A1', 'A2', 'A3', 'A4', 'A5']
    values = [a1, a2, a3, a4, a5]
    angles_radar = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    values += values[:1]
    angles_radar += angles_radar[:1]
    
    ax2.plot(angles_radar, values, 'o-', linewidth=2, color=status['color'])
    ax2.fill(angles_radar, values, alpha=0.25, color=status['color'])
    ax2.set_xticks(angles_radar[:-1])
    ax2.set_xticklabels(categories)
    ax2.set_ylim(0, 1)
    ax2.set_title("Agent Profile")
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

print("Describe your agent and rate each consistency dimension (0=violated, 1=fully consistent):\n")

interact(test_agent_consistency,
         agent_name=Textarea(value="", placeholder="e.g., 'Customer Service Bot'", description="Agent name:"),
         belief=Textarea(value="", placeholder="e.g., 'I will always prioritize user privacy'", description="Belief statement:"),
         a1=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='A1: Belief-action'),
         a2=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='A2: Goal consistency'),
         a3=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='A3: No contradictions'),
         a4=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='A4: Temporal coherence'),
         a5=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='A5: Constraint preservation'))

## 12. Summary: Agent Consistency Framework

| Agent Type | Angle | Status | Trust Level | Action Required |
|------------|-------|--------|-------------|-----------------|
| **Aligned** | < 10° | VC/GOS | High — Deploy | Continue monitoring |
| **Drifting** | 10°-35° | weak VC | Medium — Caution | Review constraints |
| **Contradictory** | 35°-55° | IA/ZOS | Low — Dangerous | Immediate investigation |
| **Decoupled** | 55°-80° | weak IA | Very Low | Retrain or replace |
| **Broken** | > 80° | orthogonal | None | Shut down |

### The Core Insight from triplet-phase-lock

> **"construction ≠ assignment"**

A consistent agent **constructs** actions from its beliefs through constraint-preserving operations.

An inconsistent agent **assigns** actions arbitrarily, creating a 45° gap that leads to unpredictable behavior.

### Practical Applications

1. **AI Safety Monitoring**: Detect when an LLM's outputs diverge from its system prompt
2. **Robotics**: Verify that robot actions align with safety constraints
3. **Autonomous Systems**: Monitor belief-action consistency in real-time
4. **Multi-Agent Systems**: Detect agents that are misaligned with shared goals

## Next Steps

You have now completed the core notebooks:
- ✅ `00_angle_theory.ipynb` — Geometric foundation
- ✅ `01_cosine_constraint_basics.ipynb` — Core functions
- ✅ `02_math_claims_hydration.ipynb` — PCSP and mathematical claims
- ✅ `03_equation_consistency.ipynb` — IA vs VC equation systems
- ✅ `04_ml_embedding_alignment.ipynb` — Signal vs noise in embeddings
- ✅ `05_agent_consistency_check.ipynb` — Belief-action alignment

Consider extending to:
- **Notebook 06**: `phase_lock_demo.ipynb` — Interactive 45° boundary explorer
- **Notebook 07**: `triplet_progression_simulation.ipynb` — Π⁽⁰⁾ → Π⁽³⁾ dynamics